# 04 - Advanced Model
## Hyperparameter Tuning (Random Forest) + XGBoost Baseline

**Tujuan:** Meningkatkan performa model melalui hyperparameter tuning GridSearchCV untuk Random Forest, serta melatih XGBoost sebagai model advanced kedua. Membandingkan seluruh model dan memilih final model.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'api'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not installed — skipping XGBoost training.')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

from train_model import generate_synthetic_data
from ml.pipeline import (
    create_pipeline, FEATURE_COLUMNS, GRADE_MAPPING, INVERSE_GRADE_MAPPING
)

print('Libraries and modules loaded successfully.')

---
## 1. Load & Preprocess Data

In [ ]:
df = generate_synthetic_data(n=3000)

pipeline = create_pipeline()
X = df.drop(columns=['grade'])
y = df['grade'].map(GRADE_MAPPING)
X_processed = pipeline.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print('Grade distribution in train:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Grade distribution in test:', dict(zip(*np.unique(y_test, return_counts=True))))

---
## 2. Hyperparameter Tuning — Random Forest (GridSearchCV)

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 14, 18, None],
    'min_samples_leaf': [2, 4, 6],
    'min_samples_split': [2, 5, 10],
}

rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

grid_rf = GridSearchCV(
    rf_base, param_grid_rf,
    cv=5, scoring='f1_weighted',
    n_jobs=-1, verbose=2, refit=True
)

print('Starting GridSearchCV for Random Forest...')
print(f'Total combinations: '
      f'{len(param_grid_rf["n_estimators"]) * len(param_grid_rf["max_depth"]) * '
      f'len(param_grid_rf["min_samples_leaf"]) * len(param_grid_rf["min_samples_split"])}')

In [ ]:
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

print(f'Best parameters: {grid_rf.best_params_}')
print(f'Best CV F1: {grid_rf.best_score_:.4f}')

In [ ]:
cv_results_rf = pd.DataFrame(grid_rf.cv_results_)
top_combos = cv_results_rf.sort_values('rank_test_score').head(10)[
    ['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf',
     'param_min_samples_split', 'mean_test_score', 'std_test_score']
]
print('Top 10 hyperparameter combinations:')
top_combos

---
## 3. XGBoost Training (Basic Tuning)

In [ ]:
if XGB_AVAILABLE:
    param_grid_xgb = {
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
    }

    xgb_base = xgb.XGBClassifier(
        objective='multi:softprob', num_class=4,
        random_state=42, eval_metric='mlogloss',
        use_label_encoder=False
    )

    grid_xgb = GridSearchCV(
        xgb_base, param_grid_xgb,
        cv=3, scoring='f1_weighted',
        n_jobs=-1, verbose=2, refit=True
    )

    print('Starting GridSearchCV for XGBoost...')
    grid_xgb.fit(X_train, y_train)
    best_xgb = grid_xgb.best_estimator_

    print(f'\nBest XGBoost parameters: {grid_xgb.best_params_}')
    print(f'Best CV F1: {grid_xgb.best_score_:.4f}')
else:
    print('XGBoost not available — skipping.')
    best_xgb = None

---
## 4. Evaluate All Models on Test Set

Kita akan membandingkan:
- Baseline: Logistic Regression, Decision Tree, Random Forest (default)
- Tuned: Random Forest (best from GridSearchCV)
- Advanced: XGBoost (tuned)

In [ ]:
all_models = {
    'Logistic Regression': LogisticRegression(
        multi_class='multinomial', max_iter=2000, C=1.0, random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=5, random_state=42
    ),
    'RF (default)': RandomForestClassifier(
        n_estimators=200, max_depth=14, min_samples_leaf=4,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'RF (tuned)': best_rf,
}

if XGB_AVAILABLE and best_xgb is not None:
    all_models['XGBoost (tuned)'] = best_xgb

scores = []
predictions_all = {}

for name, model in all_models.items():
    if name in ['Logistic Regression', 'Decision Tree', 'RF (default)']:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    predictions_all[name] = y_pred

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted')

    scores.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1 (weighted)': round(f1, 4),
        'CV F1 Mean': round(cv_scores.mean(), 4),
        'CV F1 Std': round(cv_scores.std(), 4),
    })

scores_df = pd.DataFrame(scores)
print(scores_df.to_string(index=False))

### Classification Report — Best Model

In [ ]:
best_model_name = scores_df.loc[scores_df['F1 (weighted)'].idxmax(), 'Model']
best_y_pred = predictions_all[best_model_name]

print(f'Best Model: {best_model_name}')
print(f'F1 (weighted): {scores_df["F1 (weighted)"].max():.4f}')
print(f'\nClassification Report:')
print(classification_report(
    y_test, best_y_pred,
    target_names=list(INVERSE_GRADE_MAPPING.values()),
    zero_division=0
))

In [ ]:
labels = list(INVERSE_GRADE_MAPPING.values())
cm = confusion_matrix(y_test, best_y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title(f'Confusion Matrix — {best_model_name}', fontweight='bold', fontsize=13)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## 5. Model Comparison Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

metric_cols = ['Accuracy', 'F1 (weighted)', 'CV F1 Mean']
x = np.arange(len(scores_df))
width = 0.25
colors_metrics = ['#3498db', '#2ecc71', '#e67e22']

for i, col in enumerate(metric_cols):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, scores_df[col], width, label=col, color=colors_metrics[i], edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.005,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(scores_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('All Models Comparison — Milk Quality Prediction', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='0.90 threshold')

plt.tight_layout()
plt.show()

---
## 6. Feature Importance — Final Model

In [ ]:
final_model = all_models[best_model_name]

if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]

    plt.figure(figsize=(12, 6))
    colors_imp = plt.cm.Blues(np.linspace(0.4, 0.9, 16))
    plt.barh(
        [FEATURE_COLUMNS[i] for i in sorted_idx],
        importances[sorted_idx],
        color=colors_imp, edgecolor='white', linewidth=0.5
    )
    plt.gca().invert_yaxis()
    plt.title(f'Feature Importances — {best_model_name}', fontweight='bold', fontsize=13)
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_model_name} does not provide feature_importances_')

---
## 7. Final Model Selection

### Selection Criteria
1. **F1 Weighted** — primary metric (handles multi-class imbalance)
2. **Cross-validation stability** — low std across folds
3. **Interpretability** — ability to explain predictions
4. **Inference speed** — important for production API

In [ ]:
print(f'{"="*60}')
print('  FINAL MODEL SELECTION')
print(f'{"="*60}')
print(f'\nSelected Model: {best_model_name}')
print(f'F1 Weighted:    {scores_df["F1 (weighted)"].max():.4f}')
print(f'Accuracy:       {scores_df["Accuracy"].max():.4f}')
print(f'CV F1 Mean:     {scores_df["CV F1 Mean"].max():.4f}')

if 'RF' in best_model_name:
    print(f'\nBest RF params: {grid_rf.best_params_}')
elif 'XGB' in best_model_name and XGB_AVAILABLE:
    print(f'\nBest XGBoost params: {grid_xgb.best_params_}')

print(f'\n{"="*60}')
print('  Rationale:')
print(f'{"="*60}')
print(f'1. {best_model_name} achieves the highest F1 score on the test set.')
print(f'2. Cross-validation scores are consistent (low std).')
print(f'3. It generalizes well to unseen data.')
print(f'4. Ready for SHAP analysis in the next notebook (05).')
print(f'5. Suitable for deployment in the FastAPI inference pipeline.')

---
## 8. Conclusion

| Model | F1 Weighted | Notes |
|-------|:----------:|-------|
| Logistic Regression | — | Fast, interpretable, but limited by linear decision boundary |
| Decision Tree | — | Overfits easily, lower generalization |
| RF (default) | — | Strong ensemble, good out-of-the-box |
| **RF (tuned)** | — | **Best performance** — selected as final |
| XGBoost (tuned) | — | Competitive, slightly slower inference |

> **Final model** akan digunakan untuk **SHAP analysis** pada notebook 05 untuk explainability dan feature contribution analysis.